# MatShrink V-O precision: experimental supplement

This notebook reproduces the Experiments section of the MatShrink paper for the V-O track. Three experiments run end-to-end on a single A100 GPU:

1. **Experiment A: full WikiText-2 on SmolLM2-135M and GPT-2-124M** (`d_head = 64`). Measures condition numbers and perplexity at fp32 / fp16 / bf16 for both identity-`M` and pivoted-QR `M` selection.
2. **Experiment B: Gemma 3** (`d_head = 256`, the paper's sweet spot per Table 1). Runs on Gemma-3-1B and Gemma-3-4B. Uses Unsloth mirrors to avoid HF gating.
3. **Experiment C: MLA audit.** Downloads `config.json` and the safetensors indices for MiniCPM3-4B and DeepSeek-V2-Lite; checks whether `q_a_layernorm` / `kv_a_layernorm` tensors sit between the down- and up-projection on the MLA paths (which would block MatShrink).

**Expected wall-clock:** roughly 30 to 45 min total on A100.

**Outputs:** all results saved to `matShrink_precision_results.json` and auto-downloaded at the end.

## 1. Background: V-O MatShrink and the precision problem

**The rewrite.** Inside every attention block, the per-head value projection `W_V` and output projection `W_O` are back-to-back with only linear operators between them. Their composite `W_V · W_O` therefore has rank at most `d_head`. V-O MatShrink exploits this: split `W_O` into a square `r × r` block `M` plus the remainder, push `M` into `W_V` as `W_V → W_V · M`, and rewrite the remainder with `M⁻¹`. The leading block of the new `W_O` is now the identity and can be dropped, saving `r²` weights per head per layer. At fp32 this is bit-identical.

**Where bf16 breaks.** If `M` is ill-conditioned, `M⁻¹` has large entries, and the rewritten weights `W_V · M` and `M⁻¹ · W_O` inherit that inflation. bf16's 7-bit mantissa cannot faithfully represent the inflated entries, and the error compounds across layers. The naive choice `M = W_O[:, :r]` (the leading `r` columns of `W_O`) is often badly conditioned on real weights (up to cond ≈ 10⁵ in the models tested here), which produces a bf16 perplexity regression of up to +25 absolute relative to the source model.

**The fix.** The two back-to-back matrices meet along a shared `r`-dimensional axis (the head dimension). Applying any invertible change of basis `T` to that axis as `W_V → W_V · T` and `W_O → T⁻¹ · W_O` leaves the composite unchanged, but changes *which* `r × r` sub-block of `W_O` ends up in MatShrink's leading position and therefore gets inverted. A pure permutation of that axis is the orthogonal special case and does *not* help, because orthogonal transforms preserve singular values and therefore preserve cond(`M`). A general invertible `T` can reduce cond(`M`) by 1 to 3 orders of magnitude. Rank-revealing (pivoted) QR on `W_O` picks the best-conditioned `r × r` sub-block directly (a one-line `scipy.linalg.qr(..., pivoting=True)` call), and that block is mathematically equivalent to running vanilla MatShrink after a specific change of basis. The rewritten model has the same parameter count, the same tensor shapes, and loads in stock HuggingFace Transformers and vLLM without modification.

## 2. Setup (A100 check + dependencies)

In [ ]:
try:
    import transformer_tricks
    import scipy
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'transformer-tricks', 'scipy', 'datasets', 'safetensors'], check=True)
    import transformer_tricks
    import scipy

import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime -> Change runtime type -> A100 GPU.")
gpu_name = torch.cuda.get_device_name(0)
gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}")
print(f"VRAM: {gpu_mem:.1f} GB")
if 'A100' not in gpu_name:
    print(f"WARNING: expected A100, got {gpu_name}. Gemma-3-4B may OOM on smaller cards; "
          f"Gemma-3-1B should still fit.")

In [ ]:
import os, gc, json, shutil
import numpy as np
import scipy.linalg
import datasets
from tqdm import tqdm
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
from transformer_tricks import get_param, save_repo
from huggingface_hub import HfApi, hf_hub_download

os.makedirs('_work', exist_ok=True)

## 3. MatShrink with selectable M-strategy

One parameterized function for each architecture family: `strategy='identity'` reproduces the default first-`r`-columns `M`-selection that vanilla MatShrink uses; `strategy='pivoted_qr'` picks the best-conditioned `r x r` sub-block of `W_O` via rank-revealing QR. Reads `head_dim` from config so Gemma 3 (explicit `head_dim=256`) works correctly.

In [ ]:
def _pick_M_rows(O_slice, head_dim, strategy):
    """Pick the head_dim row-indices of O_slice that will form M.
    O_slice shape: (d_model, head_dim). M shape will be (head_dim, head_dim).
    """
    if strategy == 'identity':
        return list(range(head_dim))
    elif strategy == 'pivoted_qr':
        # QR on the transpose so "column pivoting" acts on O_slice's rows
        arr = O_slice.cpu().numpy().T
        _, _, piv = scipy.linalg.qr(arr, pivoting=True)
        return piv[:head_dim].tolist()
    else:
        raise ValueError(f"unknown strategy: {strategy}")

def _pick_M_cols(W_O_h, head_dim, strategy):
    """Pick the head_dim column-indices of W_O_h that will form M (GPT-2 Conv1D variant).
    W_O_h shape: (head_dim, d_model). M shape will be (head_dim, head_dim)."""
    if strategy == 'identity':
        return list(range(head_dim))
    elif strategy == 'pivoted_qr':
        arr = W_O_h.cpu().numpy()
        _, _, piv = scipy.linalg.qr(arr, pivoting=True)
        return piv[:head_dim].tolist()
    else:
        raise ValueError(f"unknown strategy: {strategy}")

In [ ]:
def _text_config(config):
    """Unwrap multimodal configs (Gemma-3 4B+, Qwen3.6, DeepSeek-VL, ...) to the text subconfig."""
    return getattr(config, 'text_config', None) or config

def _detect_v_prefix(param, config=None):
    """Auto-detect which prefix the text-backbone V/O projections live under.
    If config is multimodal, prefer multimodal prefixes first to avoid matching
    cross-contaminated cached keys from a previous experiment."""
    is_multimodal = config is not None and getattr(config, 'text_config', None) is not None
    order = ['language_model.model', 'text_model.model', 'model'] if is_multimodal \
            else ['model', 'language_model.model', 'text_model.model']
    for prefix in order:
        if f'{prefix}.layers.0.self_attn.v_proj.weight' in param:
            return prefix
    raise KeyError(f"V/O projection keys not found. Sample keys: {list(param.keys())[:8]}")

def matshrink_vo_gqa(param, config, strategy='identity', verbose=True):
    """V-O MatShrink for GQA/MHA Llama-family models (SmolLM2, Gemma 3, Llama 3, ...).
    Unwraps multimodal configs and auto-detects the state-dict prefix."""
    cfg = _text_config(config)
    n_q_heads  = cfg.num_attention_heads
    n_kv_heads = getattr(cfg, 'num_key_value_heads', n_q_heads)
    n_groups   = n_q_heads // n_kv_heads
    head_dim   = getattr(cfg, 'head_dim', None) or (cfg.hidden_size // n_q_heads)
    n_layers   = cfg.num_hidden_layers
    prefix = _detect_v_prefix(param, config)
    if prefix != 'model':
        print(f"  (multimodal model detected -- operating on text backbone at {prefix!r})")

    # Sanity check: actual V/O shapes must match config
    v0 = param[f"{prefix}.layers.0.self_attn.v_proj.weight"]
    o0 = param[f"{prefix}.layers.0.self_attn.o_proj.weight"]
    exp_v = (n_kv_heads * head_dim, cfg.hidden_size)
    exp_o = (cfg.hidden_size, n_q_heads * head_dim)
    if tuple(v0.shape) != exp_v or tuple(o0.shape) != exp_o:
        raise RuntimeError(
            f"Shape mismatch! config expects V={exp_v}, O={exp_o}; "
            f"actual V={tuple(v0.shape)}, O={tuple(o0.shape)}. "
            f"Likely a stale cache from a previous experiment -- clean ./get_param_tmp.")

    diag = {"cond_numbers": [], "max_recon_err": [], "strategy": strategy, "head_dim": head_dim}

    with torch.no_grad():
        for layer in tqdm(range(n_layers), desc=f"{strategy}", leave=False):
            v_key = f"{prefix}.layers.{layer}.self_attn.v_proj.weight"
            o_key = f"{prefix}.layers.{layer}.self_attn.o_proj.weight"
            V = param[v_key].clone().to(torch.float64)
            O = param[o_key].clone().to(torch.float64)

            for kv_idx in range(n_kv_heads):
                v_row_start = kv_idx * head_dim
                v_row_end   = v_row_start + head_dim
                V_g = V[v_row_start:v_row_end, :]
                first_q_head = kv_idx * n_groups
                o_col_start = first_q_head * head_dim
                o_col_end   = o_col_start + head_dim
                O_h_first = O[:, o_col_start:o_col_end]

                M_rows = _pick_M_rows(O_h_first, head_dim, strategy)
                M = O_h_first[M_rows, :].clone()
                cond = torch.linalg.cond(M).item()
                diag["cond_numbers"].append(cond)
                if cond > 1e8:
                    raise RuntimeError(f"Layer {layer}, kv {kv_idx}: cond={cond:.2e} -- abort")
                M_inv = torch.linalg.inv(M)

                V[v_row_start:v_row_end, :] = M @ V_g
                for q_head in range(kv_idx * n_groups, (kv_idx + 1) * n_groups):
                    c0 = q_head * head_dim
                    c1 = c0 + head_dim
                    O[:, c0:c1] = O[:, c0:c1] @ M_inv

                identity_check = O[M_rows, :][:, o_col_start:o_col_end]
                err = (identity_check - torch.eye(head_dim, dtype=torch.float64)).abs().max().item()
                diag["max_recon_err"].append(err)

            param[v_key] = V.to(torch.float32).contiguous()
            param[o_key] = O.to(torch.float32).contiguous()

    if verbose:
        c = np.array(diag["cond_numbers"]); e = np.array(diag["max_recon_err"])
        print(f"[{strategy}] {len(c)} pairs  cond: median={np.median(c):.2e} max={np.max(c):.2e}  |  identity err max={np.max(e):.2e}")
    return diag

In [ ]:
def matshrink_vo_gpt2(param, config, strategy='identity', verbose=True):
    """GPT-2 Conv1D variant. `h.{i}.attn.c_attn.weight` / `c_proj.weight` naming."""
    d_model  = config.n_embd
    n_heads  = config.n_head
    head_dim = d_model // n_heads
    n_layers = config.n_layer
    v_offset = 2 * d_model

    diag = {"cond_numbers": [], "max_recon_err": [], "strategy": strategy, "head_dim": head_dim}

    with torch.no_grad():
        for layer in tqdm(range(n_layers), desc=f"{strategy}", leave=False):
            cattn_w_key = f"h.{layer}.attn.c_attn.weight"
            cattn_b_key = f"h.{layer}.attn.c_attn.bias"
            cproj_w_key = f"h.{layer}.attn.c_proj.weight"
            cattn_w = param[cattn_w_key].clone().to(torch.float64)
            cattn_b = param[cattn_b_key].clone().to(torch.float64)
            cproj_w = param[cproj_w_key].clone().to(torch.float64)

            for h in range(n_heads):
                v_c0 = v_offset + h * head_dim
                v_c1 = v_offset + (h + 1) * head_dim
                o_r0 = h * head_dim
                o_r1 = (h + 1) * head_dim
                W_V_h = cattn_w[:, v_c0:v_c1]
                b_V_h = cattn_b[v_c0:v_c1]
                W_O_h = cproj_w[o_r0:o_r1, :]

                M_cols = _pick_M_cols(W_O_h, head_dim, strategy)
                M = W_O_h[:, M_cols].clone()
                cond = torch.linalg.cond(M).item()
                diag["cond_numbers"].append(cond)
                if cond > 1e8:
                    raise RuntimeError(f"Layer {layer}, head {h}: cond={cond:.2e}: abort")
                M_inv = torch.linalg.inv(M)

                cattn_w[:, v_c0:v_c1] = W_V_h @ M
                cattn_b[v_c0:v_c1]    = b_V_h @ M
                cproj_w[o_r0:o_r1, :] = M_inv @ W_O_h

                identity_check = cproj_w[o_r0:o_r1, :][:, M_cols]
                err = (identity_check - torch.eye(head_dim, dtype=torch.float64)).abs().max().item()
                diag["max_recon_err"].append(err)

            param[cattn_w_key] = cattn_w.to(torch.float32).contiguous()
            param[cattn_b_key] = cattn_b.to(torch.float32).contiguous()
            param[cproj_w_key] = cproj_w.to(torch.float32).contiguous()

    if verbose:
        c = np.array(diag["cond_numbers"]); e = np.array(diag["max_recon_err"])
        print(f"[{strategy}] {len(c)} pairs  cond: median={np.median(c):.2e} max={np.max(c):.2e}  |  identity err max={np.max(e):.2e}")
    return diag

In [ ]:
def perplexity_gpu(repo, speedup=1, max_length=2048, dtype=torch.float16):
    """Perplexity on WikiText-2-raw test split with non-overlapping windows.

    `speedup` is a debug subset-divisor: `seq_len` is truncated to
    `total_tokens // speedup`, so `speedup=1` evaluates the full test split
    (paper numbers) and e.g. `speedup=16` evaluates 1/16th of it for quick
    iteration. Every cell below passes `speedup=1`.

    Auto-caps `max_length` at each model's context window. Handles multimodal
    models by reading `text_config` and, if needed, loading via
    `AutoModelForImageTextToText` and running on the language backbone."""
    cfg = AutoConfig.from_pretrained(repo)
    text_cfg = getattr(cfg, 'text_config', None) or cfg
    model_ctx = getattr(text_cfg, 'n_positions', None) or getattr(text_cfg, 'max_position_embeddings', 2048)
    max_length = min(max_length, model_ctx)

    tok = AutoTokenizer.from_pretrained(repo)
    try:
        model = AutoModelForCausalLM.from_pretrained(repo, torch_dtype=dtype, low_cpu_mem_usage=True).to("cuda").eval()
    except Exception as e:
        # Multimodal -- load the full model, then run perplexity on its language backbone.
        try:
            from transformers import AutoModelForImageTextToText
            full = AutoModelForImageTextToText.from_pretrained(repo, torch_dtype=dtype, low_cpu_mem_usage=True).to("cuda").eval()
        except Exception:
            from transformers import AutoModel
            full = AutoModel.from_pretrained(repo, torch_dtype=dtype, low_cpu_mem_usage=True).to("cuda").eval()
        # Try common text-backbone attribute names
        lang = None
        for attr in ('language_model', 'text_model', 'model'):
            if hasattr(full, attr):
                lang = getattr(full, attr); break
        if lang is None:
            raise RuntimeError(f"Could not find text backbone in {type(full).__name__} (orig error: {e})")
        model = lang

    test = datasets.load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    enc = tok("\n\n".join(test["text"]), return_tensors="pt")

    stride = max_length
    seq_len = enc.input_ids.size(1) // speedup
    nlls, prev_end = [], 0
    for begin in tqdm(range(0, seq_len, stride), leave=False):
        end = min(begin + max_length, seq_len)
        trg_len = end - prev_end
        input_ids  = enc.input_ids[:, begin:end].to("cuda")
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100
        with torch.no_grad():
            out = model(input_ids, labels=target_ids)
        nlls.append(out.loss.float())
        prev_end = end
        if end == seq_len:
            break
    ppl = torch.exp(torch.stack(nlls).mean()).item()
    del model; torch.cuda.empty_cache(); gc.collect()
    return ppl

In [ ]:
def _wipe_get_param_tmp():
    """Work around transformer-tricks issue #9: flashify_repo/get_param use a hardcoded
    ./get_param_tmp that accumulates safetensors across runs -> cross-model contamination.
    Clean it before every get_param() call so each experiment sees only its own weights."""
    for d in ('./get_param_tmp', 'get_param_tmp'):
        if os.path.exists(d):
            shutil.rmtree(d)

def run_experiment(repo, label, speedup=1, arch='gqa'):
    """Shrink baseline + pivoted, then measure perplexity at fp32/fp16/bf16 for each."""
    out_base = f"./_work/{label}_baseline"
    out_piv  = f"./_work/{label}_pivoted"
    for d in (out_base, out_piv):
        if os.path.exists(d):
            shutil.rmtree(d)

    fn = matshrink_vo_gqa if arch == 'gqa' else matshrink_vo_gpt2

    print(f"\n=== {label} :: shrinking (baseline) ===")
    _wipe_get_param_tmp()
    config = AutoConfig.from_pretrained(repo)
    param = get_param(repo)
    diag_b = fn(param, config, strategy='identity')
    save_repo(repo, param, config, out_base)
    del param; gc.collect()

    print(f"=== {label} :: shrinking (pivoted) ===")
    _wipe_get_param_tmp()
    param = get_param(repo)
    diag_p = fn(param, config, strategy='pivoted_qr')
    save_repo(repo, param, config, out_piv)
    del param; gc.collect()

    print(f"=== {label} :: perplexity ===")
    ppl = {}
    for name, path in [("source", repo), ("identity", out_base), ("pivoted", out_piv)]:
        for dtn, dt in [("fp32", torch.float32), ("fp16", torch.float16), ("bf16", torch.bfloat16)]:
            key = f"{name}-{dtn}"
            ppl[key] = perplexity_gpu(path, speedup=speedup, dtype=dt)
            print(f"  {key:<20s}  ppl = {ppl[key]:.4f}")

    for d in (out_base, out_piv):
        shutil.rmtree(d, ignore_errors=True)

    return {
        "repo": repo,
        "arch": arch,
        "speedup": speedup,
        "head_dim": diag_b["head_dim"],
        "n_pairs": len(diag_b["cond_numbers"]),
        "cond_identity": {"median": float(np.median(diag_b["cond_numbers"])), "max": float(np.max(diag_b["cond_numbers"])), "p95": float(np.quantile(diag_b["cond_numbers"], 0.95))},
        "cond_pivoted":  {"median": float(np.median(diag_p["cond_numbers"])), "max": float(np.max(diag_p["cond_numbers"])), "p95": float(np.quantile(diag_p["cond_numbers"], 0.95))},
        "perplexity": ppl,
    }

all_results = {}

## 4. Experiment A: full WikiText-2 on SmolLM2-135M and GPT-2-124M

Perplexity at fp32 / fp16 / bf16 across both `M`-selection strategies (identity vs pivoted-QR), evaluated on the full WikiText-2 test split. fp32 is bit-identical across strategies (algebraic invariance); the bf16 regression with identity-`M` is exactly what the pivoted-QR selection fixes.

In [ ]:
all_results["smollm2_full"] = run_experiment("HuggingFaceTB/SmolLM2-135M", "SmolLM2-135M_full", speedup=1, arch='gqa')
# Partial save after each experiment -- if a later cell OOMs or errors,
# earlier results are preserved.
with open("matShrink_precision_results.json", "w") as f:
    json.dump(all_results, f, indent=2)
print(f"(saved partial results, {len(all_results)} experiments so far)")


In [ ]:
all_results["gpt2_full"] = run_experiment("gpt2", "gpt2-124M_full", speedup=1, arch='gpt2')
# Partial save after each experiment -- if a later cell OOMs or errors,
# earlier results are preserved.
with open("matShrink_precision_results.json", "w") as f:
    json.dump(all_results, f, indent=2)
print(f"(saved partial results, {len(all_results)} experiments so far)")


## 5. Experiment B: Gemma 3 (d_head=256, the paper's sweet spot)

`d_head=256` is what makes MatShrink's `r² · h` savings meaningful at the attention-weight level (paper Table 1 lists CodeGemma-7B at 8%, T5-11B at 13%: all `d_head ≥ 128`). Using Unsloth mirrors so no HF license acceptance is needed.

In [ ]:
all_results["gemma3_1b"] = run_experiment("unsloth/gemma-3-1b-pt", "Gemma-3-1B", speedup=1, arch='gqa')
# Partial save after each experiment -- if a later cell OOMs or errors,
# earlier results are preserved.
with open("matShrink_precision_results.json", "w") as f:
    json.dump(all_results, f, indent=2)
print(f"(saved partial results, {len(all_results)} experiments so far)")


In [ ]:
# Gemma-3-4B: ~8 GB model, ~20 GB temp disk + VRAM during the three-precision sweep.
# Should fit on A100 40 GB. If VRAM-constrained, skip this cell and only run the 1B variant above.
all_results["gemma3_4b"] = run_experiment("unsloth/gemma-3-4b-pt", "Gemma-3-4B", speedup=1, arch='gqa')
# Partial save after each experiment -- if a later cell OOMs or errors,
# earlier results are preserved.
with open("matShrink_precision_results.json", "w") as f:
    json.dump(all_results, f, indent=2)
print(f"(saved partial results, {len(all_results)} experiments so far)")


## 6. Experiment C: MLA audit

MatShrink relies on the algebraic identity `W_A · W_B = (W_A · M) · (M⁻¹ · W_B)`, which only holds when the two matrices are truly back-to-back with linear operators between them. If a nonlinearity (RMSNorm / LayerNorm) sits on the shared inner dimension, the identity breaks. In MLA models, `q_a_layernorm` and `kv_a_layernorm` are exactly such in-between normalizations on the latent → up-projection pair, so their presence blocks vanilla MatShrink on that path. This cell pulls each model's `config.json` and safetensors index (not the full weights) and greps the tensor names to flag eligibility.

Target models (from paper Table 2):
- `openbmb/MiniCPM3-4B`: smallest MLA model, `r_Q=768`, `r_KV=256`, `h=40`.
- `deepseek-ai/DeepSeek-V2-Lite`: the 16B variant, `r_KV=512`, `h=16`.

In [ ]:
def audit_mla(repo):
    """Fetch config + safetensors index (if present) and report MatShrink-eligibility."""
    api = HfApi()
    try:
        cfg_path = hf_hub_download(repo, 'config.json')
        with open(cfg_path) as f:
            cfg = json.load(f)
    except Exception as e:
        return {"repo": repo, "error": f"config fetch failed: {e}"}

    # Try to pull the safetensors index; if model is single-shard there won't be one
    try:
        idx_path = hf_hub_download(repo, 'model.safetensors.index.json')
        with open(idx_path) as f:
            idx = json.load(f)
        keys = list(idx['weight_map'].keys())
    except Exception:
        keys = None  # single-shard or index missing; fall back to flagging via config

    result = {
        "repo": repo,
        "architectures": cfg.get('architectures'),
        "model_type": cfg.get('model_type'),
        "q_lora_rank": cfg.get('q_lora_rank'),
        "kv_lora_rank": cfg.get('kv_lora_rank'),
        "q_a_layernorm_keys": [],
        "kv_a_layernorm_keys": [],
        "any_a_layernorm_keys": [],
    }
    if keys is not None:
        result["q_a_layernorm_keys"]   = sorted({k for k in keys if 'q_a_layernorm' in k})[:3]
        result["kv_a_layernorm_keys"]  = sorted({k for k in keys if 'kv_a_layernorm' in k})[:3]
        result["any_a_layernorm_keys"] = sorted({k for k in keys if '_a_layernorm' in k or '_a_norm' in k})[:5]
        result["sample_attn_keys"]     = sorted({k for k in keys if '.self_attn.' in k and '.0.' in k})[:15]

    # Eligibility verdict
    has_q_norm  = bool(result["q_a_layernorm_keys"])
    has_kv_norm = bool(result["kv_a_layernorm_keys"])
    result["matshrink_q_path"]  = "BLOCKED by q_a_layernorm"  if has_q_norm  else "eligible (check manually)"
    result["matshrink_kv_path"] = "BLOCKED by kv_a_layernorm" if has_kv_norm else "eligible (check manually)"
    return result

for repo in ["openbmb/MiniCPM3-4B", "deepseek-ai/DeepSeek-V2-Lite"]:
    print(f"\n=== {repo} ===")
    r = audit_mla(repo)
    for k, v in r.items():
        if isinstance(v, list) and len(v) > 0:
            print(f"  {k}:")
            for item in v: print(f"    - {item}")
        else:
            print(f"  {k}: {v}")
    all_results.setdefault("mla_audit", {})[repo] = r

## 7. Summary tables

Per-model digest: cond(`M`) with identity vs pivoted-QR selection, fp32 and bf16 perplexity for both strategies, and the bf16 gap to source. This is the data used for the tables in `experiments.tex`.

In [ ]:
def fmt_exp(label, r):
    src32 = r["perplexity"]["source-fp32"]
    idt32 = r["perplexity"]["identity-fp32"]
    piv32 = r["perplexity"]["pivoted-fp32"]
    src16b = r["perplexity"]["source-bf16"]
    idt16b = r["perplexity"]["identity-bf16"]
    piv16b = r["perplexity"]["pivoted-bf16"]
    print(f"\n--- {label} (d_head={r['head_dim']}, n_pairs={r['n_pairs']}, speedup={r['speedup']}) ---")
    print(f"  cond(M) max:    identity={r['cond_identity']['max']:.2e}   pivoted={r['cond_pivoted']['max']:.2e}   improvement={r['cond_identity']['max']/r['cond_pivoted']['max']:.1f}x")
    print(f"  fp32 ppl:       source={src32:.4f}   identity={idt32:.4f}   pivoted={piv32:.4f}")
    print(f"  bf16 ppl:       source={src16b:.4f}   identity={idt16b:.4f}   pivoted={piv16b:.4f}")
    print(f"  bf16 gap:       identity={idt16b-src16b:+.4f}   pivoted={piv16b-src16b:+.4f}")

for lbl, key in [("SmolLM2-135M (full)","smollm2_full"),
                  ("GPT-2-124M (full)","gpt2_full"),
                  ("Gemma-3-1B","gemma3_1b"),
                  ("Gemma-3-4B","gemma3_4b")]:
    if key in all_results:
        fmt_exp(lbl, all_results[key])

In [ ]:
# Save + auto-download
with open("matShrink_precision_results.json", "w") as f:
    json.dump(all_results, f, indent=2)
print("Saved -> matShrink_precision_results.json")

try:
    from google.colab import files
    files.download("matShrink_precision_results.json")
except ImportError:
    print("Not in Colab: results are in the working directory.")

## 8. Conclusion

We ran the pivoted-QR `M`-selection experiment on four models at full WikiText-2. The pattern is consistent across all of them:

| Model | `d_head` | cond(M) max: identity -> pivoted | bf16 ppl gap to source: identity -> pivoted |
|---|---|---|---|
| SmolLM2-135M (GQA) | 64  | 9,865 -> 104     | +1.51  -> +0.004 |
| GPT-2-124M (MHA)   | 64  | 167,117 -> 404   | +18.94 -> -0.02  |
| Gemma-3-1B (GQA)   | 256 | 99,098 -> 702    | +25.29 -> -0.04  |
| Gemma-3-4B (GQA)   | 256 | 158,464 -> 5,359 | +8.98  -> +0.02  |

**Findings.**

1. **The bf16 precision regression is entirely eliminated by pivoted-QR `M` selection.** The gap from source bf16 perplexity drops from as much as +25 absolute (identity-`M`) to within +/- 0.05 on every model. fp32 numbers are bit-identical across baseline and pivoted, as required by algebraic invariance.

2. **The effect scales with head dimension, as predicted.** At `d_head = 256` the identity-`M` regression is much larger than at `d_head = 64` (bigger `r x r` block means heavier condition-number tail). Pivoted QR closes the gap cleanly in both regimes.

3. **Parameter savings are unchanged.** Pivoted QR picks a different `r x r` sub-block as `M`, but still exactly one sub-block. The identity-block saving of `r^2` weights per head per layer is identical between identity-`M` and pivoted-`M`.

4. **fp16 is already fine without any fix.** The identity-`M` variant at fp16 has a regression of +0.01 to +0.06 absolute on both `d_head = 64` models, which is within benchmark noise. The precision problem is specifically a bf16 phenomenon, driven by bf16's reduced (7-bit) mantissa. fp16's 10-bit mantissa is enough to absorb the rewritten-weight magnitudes even without the conditioning fix.

5. **MLA audit.** MiniCPM3-4B has neither `q_a_layernorm` nor `kv_a_layernorm` in its state dict, so both MatShrink paths are directly eligible. DeepSeek-V2-Lite has `kv_a_layernorm` on every layer (standard DeepSeek MLA pattern), blocking the KV path.

**Takeaway.** With pivoted-QR `M` selection, V-O MatShrink is lossless at bf16 storage (within benchmark noise) in addition to the original fp32 guarantee, extending the set of deployment formats in which the rewrite is safe.